# День 4 — Улучшение baseline-модели

**Цель:** проверить три понятные гипотезы улучшения и выбрать модель без подглядывания в test set. Baseline macro F1 дня 3 равен примерно `0.7561`; целевой относительный прирост 5% означает macro F1 не ниже `0.7939`.

In [1]:
import json
from pathlib import Path

from IPython.display import Markdown, display

from finnews_sentiment.data.load_data import DEFAULT_CONFIG, load_financial_phrasebank
from finnews_sentiment.features.preprocess import prepare_news_data
from finnews_sentiment.models.train_improved import (
    run_improvement_experiments,
    save_improvement_artifacts,
)

## 1. Конфигурация

Используем то же стратифицированное разбиение 80/20 и `random_state=42`, что и в baseline. Для выбора параметров применяется пятифолдовая стратифицированная cross-validation только на train-части.

In [3]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'pyproject.toml').exists() else current_dir.parent
assert (PROJECT_ROOT / 'pyproject.toml').exists(), 'Запустите ноутбук из корня проекта или папки notebooks'

MODEL_PATH = PROJECT_ROOT / 'models/best_model.joblib'
DAY3_METRICS_PATH = PROJECT_ROOT / 'reports/day03/baseline_metrics.json'
COMPARISON_PATH = PROJECT_ROOT / 'reports/day04/model_comparison.csv'
METRICS_PATH = PROJECT_ROOT / 'reports/day04/improvement_metrics.json'
TEST_SIZE = 0.2
RANDOM_STATE = 42
CV_SPLITS = 5

## 2. Данные и baseline

Ноутбук автономно загружает Financial PhraseBank и применяет тот же preprocessing, что и предыдущие этапы. Baseline переобучается на том же split, поэтому сравнение остаётся честным.

In [4]:
raw_df = load_financial_phrasebank(DEFAULT_CONFIG)
df = prepare_news_data(raw_df)
print(f'Исходных строк: {len(raw_df)}')
print(f'После preprocessing: {len(df)}')
display(df[['text_clean', 'sentiment']].head())

Исходных строк: 3453
После preprocessing: 3448


,text_clean,sentiment
0,"according to gran , the company has no plans t...",neutral
1,with the new production plant the company woul...,positive
2,"for the last quarter of 2010 , componenta 's n...",positive
3,"in the third quarter of 2010 , net sales incre...",positive
4,operating profit rose to eur 13.1 mn from eur ...,positive


## 3. Три гипотезы улучшения

1. **Настройка TF-IDF:** сравним униграммы, биграммы и триграммы, порог редких слов, ограничение словаря, sublinear TF и силу регуляризации `C`.
2. **Балансировка классов:** проверим `class_weight='balanced'`. Это даёт редким классам больший вес и часто улучшает macro F1.
3. **LinearSVC:** линейный SVM обычно хорошо работает с разреженными высокоразмерными текстовыми признаками.

Каждый TF-IDF находится внутри `Pipeline`, поэтому он обучается заново внутри каждого CV fold и не видит validation/test тексты.

In [5]:
best_model, comparison, metadata = run_improvement_experiments(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    cv_splits=CV_SPLITS,
    n_jobs=-1,
)

In [5]:
day3_metrics = json.loads(DAY3_METRICS_PATH.read_text(encoding='utf-8'))
assert abs(metadata['baseline_macro_f1'] - day3_metrics['macro_f1']) < 1e-12
print(f"Baseline воспроизведён: macro F1 = {metadata['baseline_macro_f1']:.4f}")

Baseline воспроизведён: macro F1 = 0.7561


## 4. Сравнение результатов

`cv_macro_f1` используется для выбора победителя. Test-колонки показывают итоговую обобщающую способность и не участвуют в выборе. `relative_gain_percent` считается относительно test macro F1 baseline.

In [6]:
display(
    comparison.drop(columns='best_params').style.format({
        'cv_macro_f1': '{:.4f}',
        'test_accuracy': '{:.4f}',
        'test_macro_f1': '{:.4f}',
        'test_weighted_f1': '{:.4f}',
        'absolute_gain': '{:+.4f}',
        'relative_gain_percent': '{:+.2f}%',
    })
)

,method,cv_macro_f1,test_accuracy,test_macro_f1,test_weighted_f1,absolute_gain,relative_gain_percent,selected
0,baseline_logreg,0.7226,0.8333,0.7561,0.8204,+0.0000,+0.00%,False
1,tuned_tfidf_logreg,0.7871,0.8565,0.7936,0.8492,+0.0375,+4.95%,False
2,balanced_logreg,0.7982,0.8478,0.8000,0.8455,+0.0439,+5.81%,False
3,linear_svc,0.8070,0.8623,0.8096,0.8577,+0.0535,+7.07%,True


In [7]:
for experiment in metadata['experiments']:
    print(f"{experiment['method']}: {experiment['best_params']}")

baseline_logreg: {'tfidf__max_features': 5000, 'tfidf__ngram_range': [1, 2], 'clf__C': 1.0, 'clf__class_weight': None}
tuned_tfidf_logreg: {'clf__C': 5.0, 'tfidf__max_features': 5000, 'tfidf__min_df': 2, 'tfidf__ngram_range': [1, 1], 'tfidf__sublinear_tf': True}
balanced_logreg: {'tfidf__max_features': 5000, 'tfidf__min_df': 2, 'tfidf__ngram_range': [1, 1], 'tfidf__sublinear_tf': True, 'clf__C': 5.0, 'clf__class_weight': 'balanced'}
linear_svc: {'tfidf__max_features': 5000, 'tfidf__min_df': 2, 'tfidf__ngram_range': [1, 1], 'tfidf__sublinear_tf': True, 'clf__C': 0.5, 'clf__class_weight': 'balanced'}


### Что означает результат

- Рост после настройки TF-IDF показывает пользу более подходящего представления текста и регуляризации.
- Изменение после `class_weight='balanced'` показывает влияние дисбаланса классов.
- Сравнение Logistic Regression и LinearSVC отделяет влияние классификатора от влияния признаков.
- Accuracy может расти не синхронно с macro F1: macro F1 одинаково учитывает все три класса, включая редкие.

## 5. Победитель и целевой прирост

In [8]:
winner_row = comparison.loc[comparison['selected']].iloc[0]
status = 'достигнута' if metadata['target_reached'] else 'не достигнута'
display(Markdown(
    f"**Победитель по CV:** `{metadata['winner']}`  \n"
    f"**Test macro F1:** `{winner_row['test_macro_f1']:.4f}`  \n"
    f"**Целевой macro F1:** `{metadata['target_macro_f1']:.4f}`  \n"
    f"**Цель +5%:** {status}."
))

**Победитель по CV:** `linear_svc`  
**Test macro F1:** `0.8096`  
**Целевой macro F1:** `0.7939`  
**Цель +5%:** достигнута.

## 6. Сохранение артефактов

Сохраняем весь pipeline-победитель: при inference не потребуется отдельно загружать или обучать TF-IDF.

In [9]:
save_improvement_artifacts(
    best_model,
    comparison,
    metadata,
    MODEL_PATH,
    COMPARISON_PATH,
    METRICS_PATH,
)
print(f'Модель: {MODEL_PATH.relative_to(PROJECT_ROOT)}')
print(f'Сравнение: {COMPARISON_PATH.relative_to(PROJECT_ROOT)}')
print(f'Полные метрики: {METRICS_PATH.relative_to(PROJECT_ROOT)}')

Модель: models/best_model.joblib
Сравнение: reports/day04/model_comparison.csv
Полные метрики: reports/day04/improvement_metrics.json


## 7. Выводы

Победитель выбран по cross-validation, а test set использован только для финальной проверки. Это важнее небольшого дополнительного прироста, который можно было бы получить, многократно подстраиваясь под test. На дне 5 ошибки лучшей модели будут разобраны по классам и реальным примерам.